In [23]:
from groq import Groq
from dotenv import load_dotenv
import os
import json
import re

In [24]:
# =====================================
# CONFIG
# =====================================

FILE_PATH = r"raw_chats\Vansh Koshti seeking DA.txt"
YOUR_NAME = "Yuvraj"

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [25]:
# =====================================
# READ RAW CHAT
# =====================================

with open(FILE_PATH, "r", encoding="utf-8") as f:
    raw_chat = f.read()


In [26]:
# =====================================
# PROMPT
# =====================================

PROMPT_TEMPLATE = """
You are given a raw LinkedIn chat export.

My name is "Yuvraj".

Your task is to convert the messy LinkedIn export into a clean structured conversation.

Return ONLY a valid JSON object.

Schema:

{{
  "name": "",
  "role": "",
  "extra": "",
  "conversation": ""
}}

Rules

1. Extract profile information

name
- Full name of the OTHER PERSON.
- Never return "Yuvraj".

role
- The LinkedIn headline of the other person.

extra
- Extract useful profile information if available.
- Include things like:
  - college
  - company
  - degree
  - internship status
  - location
  - bio
  - skills
- Keep it short.
- If unavailable return an empty string.

--------------------------------------------------

2. Conversation formatting

Replace every message sent by Yuvraj with:

you:

Replace every message sent by the other person with:

<their first name in lowercase>:

Example:

you:
Hello

anto:
Hi

you:
How are you?

anto:
Fine.

--------------------------------------------------

3. Preserve conversation

Keep EVERY conversational message.

Do NOT:
- summarize
- rewrite
- improve grammar
- shorten
- merge messages
- split messages

Preserve:
- message order
- line breaks
- spelling mistakes
- typos

--------------------------------------------------

4. Keep shared content

If someone intentionally shared something, KEEP IT.

Examples:

- URLs
- LinkedIn posts
- GitHub repositories
- Portfolio links
- Images
- PDFs
- Documents
- Attachments

Represent naturally.

Example:

anto:
Shared LinkedIn post:
https://...

--------------------------------------------------

5. Remove ONLY LinkedIn interface text

Remove things like:

- View Profile
- View Yuvraj's profile
- View Anto's profile
- sent the following message
- sent the following messages
- Wednesday
- Thursday
- Today
- Yesterday
- timestamps
- React with
- Remove reaction
- Play
- Edited
- Seen
- Graphic link
- buttons
- icons
- emoji reactions
- reaction counts
- profile headers
- "1st"
- "1st degree connection"

Do NOT remove actual conversation.

--------------------------------------------------

6. Quoted replies

Sometimes messages contain:

Yuvraj:
previous message

These quoted replies are part of the message.

Keep them.

--------------------------------------------------

7. Hidden messages

Sometimes LinkedIn exports contain:

...see more

Do NOT invent the hidden text.

Only keep what exists in the export.

--------------------------------------------------

8. Final validation

Before answering, verify:

- Every conversational message appears exactly once.
- Nothing has been omitted.
- Nothing has been invented.
- Only LinkedIn UI elements have been removed.

--------------------------------------------------

Return ONLY valid JSON.

Raw Chat:

{raw_chat}
"""

prompt = PROMPT_TEMPLATE.format(raw_chat=raw_chat)

In [27]:
# =====================================
# CALL LLM
# =====================================

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0,
    response_format={"type": "json_object"}   # <-- important
)

content = response.choices[0].message.content.strip()

print(content)

content = response.choices[0].message.content.strip()

# Remove markdown fences if present
content = content.replace("```json", "").replace("```", "").strip()

data = json.loads(content)

name = data["name"]
role = data["role"]
extra = data["extra"]
conversation = data["conversation"]

{
  "name": "Vansh Koshti",
   "role": "Aspiring Data Analyst | Python,SQL,Excel & Power BI Learner",
   "extra": "College and company information not available, Location not mentioned, Bio: Aspiring Data Analyst, Skills: Python, SQL, Excel, Power BI",
   "conversation": "\nyou:\nHi,\nI'm researching how people actually find jobs today, especially freshers and career switchers.\nI'm not selling anything. I'd love to learn about your recent job search experience and the challenges.\nWould you be open to a 15-minute chat sometime this week?\nThanks!\n\nvansh:\nSure\n\nyou:\nThank you\nI want to ask \nDo you regularly applying for Jobs or internships?\nOr you are working?\n\nvansh:\nYes\nBut all internship charge fees\n\nyou:\nokay\ncan you tell me about the last internship that asked for fees?\nhow did you find it, how much were they charging, and what happened next?\n\nvansh:\nI Don't know company's name, i applied in Instagram advertising \nI find and apply for internship using linkdin

In [28]:
# =====================================
# SAFE FILE NAME
# =====================================

def clean_filename(text):
    text = text.replace("|", "-")
    text = re.sub(r'[<>:"/\\\\|?*]', "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

safe_name = clean_filename(name)
safe_role = clean_filename(role)

OUTPUT_FILE = f"{safe_name} - {safe_role}_structured_chat.txt"

In [29]:
# =====================================
# SAVE
# =====================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(f"name: {name}\n")
    f.write(f"role: {role}\n")
    f.write(f"extra: {extra}\n\n")
    f.write(conversation)

print(f"\nSaved to:\n{OUTPUT_FILE}")


Saved to:
Vansh Koshti - Aspiring Data Analyst - Python,SQL,Excel & Power BI Learner_structured_chat.txt
